# SLM Fine-Tuning Demo: Google Gemma 3 (270M)

This notebook demonstrates fine-tuning **Gemma 3 270M Instruct** using **LoRA (Low-Rank Adaptation)** and Hugging Face **TRL (`SFTTrainer`)** with an instruction dataset.

### Pipeline Overview
1. **Environment & Hardware Verification**
2. **Load Model & Tokenizer** (`google/gemma-3-270m-it`)
3. **Prepare Instruction Dataset** (`yahma/alpaca-cleaned`)
4. **Configure LoRA (PEFT)**
5. **Supervised Fine-Tuning (SFT)**
6. **Inference & Before/After Comparison**
7. **Save Adapters for Deployment**

In [ ]:
# Step 1: Install Dependencies
!pip install -q transformers peft trl datasets accelerate bitsandbytes

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

In [ ]:
# Step 2: Load Model & Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "google/gemma-3-270m-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in bfloat16 (or float16 depending on GPU)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto"
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Step 3: Dataset Preparation (Instruction-Following with Alpaca-Cleaned)
from datasets import load_dataset

# Load a sample of 2,000 instruction-response pairs for fast demonstration
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")

def format_prompts(batch):
    texts = []
    for instruction, input_text, output in zip(batch["instruction"], batch["input"], batch["output"]):
        user_content = f"{instruction}\n{input_text}".strip() if input_text else instruction
        messages = [
            {"role": "user", "content": user_content},
            {"role": "model", "content": output}
        ]
        # Use tokenizer chat template
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(formatted_text)
    return {"text": texts}

formatted_dataset = raw_dataset.map(format_prompts, batched=True)
print("Sample formatted prompt:")
print(formatted_dataset[0]["text"])

In [ ]:
# Step 4: Configure LoRA (PEFT)
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Training Configuration & SFTTrainer
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./gemma3-270m-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=60,          # ~1-2 minute demo run; increase for full convergence
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("Starting training...")
trainer.train()

In [ ]:
# Step 6: Test Inference with the Fine-Tuned Model
test_prompt = "Give three concise tips for maintaining high code quality in Python."
messages = [{"role": "user", "content": test_prompt}]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Model Response:")
print(response)

In [ ]:
# Step 7: Save LoRA Weights
adapter_path = "./gemma3_270m_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter successfully saved to: {adapter_path}")